---
title: "Publish: Gather the Aggregated Data and Publish to DataVerse"
---

## publish 

> This is the `publish` module for the ERA5 dataset pipeline. It defines a functions that make use of the `pyDataverse` library and API to publish our outputs to the Harvard Dataverse.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

First, we'll test out the API by pinging the Harvard DataVerse

In [ ]:
api_token_file = here() / "sandbox/dataverse_api_key.yml"
with open(api_token_file, "r") as f:
    config = yaml.load(f, Loader=yaml.BaseLoader)

Now, following the [docs]() for the dataverse tutorial, load a NativeAPI up:

The NativeAPI is a catchall API object to be able to do general stuff:

In [ ]:
api = NativeApi(config['base_url'], config['api_token'])
resp=api.get_info_version()
#resp.text()

In [ ]:
resp.json()

{'status': 'OK', 'data': {'version': '6.7', 'build': 'iqss-2'}}

Looks good! Now that we know that it works, we can think more
about how to publish data there.

## Harvard Dataverse

Let's create a dummy dataset with the components we're
planning to upload, and then upload and promptly delete it.

To do that, we must import the `models` module and create a Dataset object:

In [ ]:
from pyDataverse.models import Dataset
ds = Dataset()

This `ds` object is pretty straightforward since it doesn't contain anything yet:

In [ ]:
ds.get()

{}

We can populate the object from the dummy data on the github repo:

In [ ]:
from pyDataverse.utils import read_file
from urllib.request import urlretrieve
import tempfile

# url for dummy data
url = "https://raw.githubusercontent.com/gdcc/pyDataverse/refs/heads/main/tests/data/user-guide/dataset.json"


with tempfile.NamedTemporaryFile(mode='w+') as tmp:
    urlretrieve(url, tmp.name)
    ds.from_json(read_file(tmp.name))

We have to validate the JSON correctly:

In [ ]:
ds.validate_json()

True

Modifying it is easy:

In [ ]:
ds.set({"title": "Youth from Austria 2005"})
ds.get()

{'citation_displayName': 'Citation Metadata',
 'title': 'Youth from Austria 2005',
 'author': [{'authorName': 'LastAuthor1, FirstAuthor1',
   'authorAffiliation': 'AuthorAffiliation1'}],
 'datasetContact': [{'datasetContactEmail': 'ContactEmail1@mailinator.com',
   'datasetContactName': 'LastContact1, FirstContact1'}],
 'dsDescription': [{'dsDescriptionValue': 'DescriptionText'}],
 'subject': ['Medicine, Health and Life Sciences']}

Now, to create the dataset we use the API:

In [ ]:
#| eval: false
resp = api.create_dataset(":root", ds.json())

Dataset with pid 'doi:10.7910/DVN/QDNSV8' created.


If you caught the `resp` object, it contains the PID for the newly created dataset.

However, if you didn't you can use the SearchAPI to find it:

In [ ]:
#| eval: false
from pyDataverse.api import SearchApi

search_api = SearchApi(config['base_url'], config['api_token'])
resp = search_api.search("Youth from Austria", data_type="dataset")
results = resp.json()['data']['items']
result = [x for x in results if "Youth from Austria" in x['name']][0]
result

{'name': 'Youth from Austria 2005',
 'type': 'dataset',
 'url': 'https://doi.org/10.7910/DVN/SUVFBV',
 'global_id': 'doi:10.7910/DVN/SUVFBV',
 'description': 'DescriptionText',
 'publisher': 'Harvard Dataverse',
 'citationHtml': 'LastAuthor1, FirstAuthor1, 2025, "Youth from Austria 2005", <a href="https://doi.org/10.7910/DVN/SUVFBV" target="_blank">https://doi.org/10.7910/DVN/SUVFBV</a>, Harvard Dataverse, DRAFT VERSION',
 'identifier_of_dataverse': 'harvard',
 'name_of_dataverse': 'Harvard Dataverse',
 'citation': 'LastAuthor1, FirstAuthor1, 2025, "Youth from Austria 2005", https://doi.org/10.7910/DVN/SUVFBV, Harvard Dataverse, DRAFT VERSION',
 'publicationStatuses': ['Unpublished', 'Draft'],
 'storageIdentifier': 's3://10.7910/DVN/SUVFBV',
 'subjects': ['Medicine, Health and Life Sciences'],
 'fileCount': 0,
 'versionId': 499516,
 'versionState': 'DRAFT',
 'createdAt': '2025-07-31T23:31:16Z',
 'updatedAt': '2025-07-31T23:31:16Z',
 'contacts': [{'name': 'LastContact1, FirstContact1', 

In [ ]:
#| eval: false
pid = result['global_id']

Now to look at the data we created using the NativeAPI again, and delete the dataset:

In [ ]:
#| eval: false
uploaded_ds = api.get_dataset(pid)
uploaded_ds.json()['data']

resp = api.delete_dataset(pid)
resp.json()

Dataset 'doi:10.7910/DVN/SUVFBV' deleted.


{'status': 'OK', 'data': {'message': 'Dataset :persistentId deleted'}}

With that understanding, we can develop a quick module to do the following:

1. Make the dataset LEGO Compatible
2. Upload and publish the data to dataverse

## LEGO Compatibility

Let's take an example file to use as a model for LEGO compatibility

In [ ]:
ex = gpd.read_parquet(here() / "data" / "testing" / "madagascar_environmental_exposure-era5_healthshed_2m_dewpoint_temperature_2009_6.parquet")
ex.describe()

,day_01_daily_mean,day_02_daily_mean,day_03_daily_mean,day_04_daily_mean,day_05_daily_mean,day_06_daily_mean,day_07_daily_mean,day_08_daily_mean,day_09_daily_mean,day_10_daily_mean,day_11_daily_mean,day_12_daily_mean,day_13_daily_mean,day_14_daily_mean,day_15_daily_mean,day_16_daily_mean,day_17_daily_mean,day_18_daily_mean,day_19_daily_mean,day_20_daily_mean,day_21_daily_mean,day_22_daily_mean,day_23_daily_mean,day_24_daily_mean,day_25_daily_mean,day_26_daily_mean,day_27_daily_mean,day_28_daily_mean,day_29_daily_mean,day_30_daily_mean,day_01_daily_min,day_02_daily_min,day_03_daily_min,day_04_daily_min,day_05_daily_min,day_06_daily_min,day_07_daily_min,day_08_daily_min,day_09_daily_min,day_10_daily_min,day_11_daily_min,day_12_daily_min,day_13_daily_min,day_14_daily_min,day_15_daily_min,day_16_daily_min,day_17_daily_min,day_18_daily_min,day_19_daily_min,day_20_daily_min,day_21_daily_min,day_22_daily_min,day_23_daily_min,day_24_daily_min,day_25_daily_min,day_26_daily_min,day_27_daily_min,day_28_daily_min,day_29_daily_min,day_30_daily_min,day_01_daily_max,day_02_daily_max,day_03_daily_max,day_04_daily_max,day_05_daily_max,day_06_daily_max,day_07_daily_max,day_08_daily_max,day_09_daily_max,day_10_daily_max,day_11_daily_max,day_12_daily_max,day_13_daily_max,day_14_daily_max,day_15_daily_max,day_16_daily_max,day_17_daily_max,day_18_daily_max,day_19_daily_max,day_20_daily_max,day_21_daily_max,day_22_daily_max,day_23_daily_max,day_24_daily_max,day_25_daily_max,day_26_daily_max,day_27_daily_max,day_28_daily_max,day_29_daily_max,day_30_daily_max
count,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000
mean,290.101105,290.251129,290.299927,290.669952,290.294189,289.835541,289.162964,288.179321,287.109406,287.835236,288.028015,288.088745,288.148865,288.321899,288.030060,287.259186,287.352325,286.910736,287.242310,286.907074,286.700073,287.705170,288.252197,288.309204,287.581604,286.087097,285.512421,285.862244,285.073608,281.498901,288.446167,288.635742,288.183105,289.243347,288.659363,288.219482,287.548737,286.358459,285.077789,286.441223,286.392883,286.434753,286.206726,286.477905,286.038940,285.196930,285.440979,284.557495,285.394012,285.095520,284.490295,285.589783,286.482513,286.632172,285.370239,283.259369,282.805817,283.903046,282.462799,278.705475,292.094940,292.087097,292.308136,292.333740,292.044861,291.473663,290.874512,289.941895,288.923279,289.482208,289.540985,289.801086,290.197021,290.187256,289.883636,289.137390,289.048706,288.969818,288.985046,288.425049,288.465210,289.768005,290.134491,290.183838,289.658630,288.893921,288.319275,287.971619,287.961121,284.683014
std,3.746835,3.516243,3.244272,2.700433,2.922641,3.155868,2.765681,3.065981,3.427631,2.879560,2.653996,2.929678,3.225507,3.233978,3.314857,3.518277,3.192194,3.181464,2.949577,3.010527,3.238273,3.158803,3.063348,3.162030,3.282480,4.012838,3.678059,3.925954,4.494281,4.591322,4.253387,3.812573,4.040436,3.007502,3.312206,3.748856,3.125356,3.545921,4.025942,3.1

We know that the LEGO data model should look like this:

```
<main lab folder>/lego
├── <domain>
│   ├── <subdomain>__<data_source>
│   │   ├── <geo_resolution>__<time_resolution>
│   │   │   ├── <filename>_yyyy.parquet
```

So, for the above file, we'll end up with the LEGO path `data/environmental/exposures_era5/healthshed_monthly/dewpoint_2024.parquet`. In it, we should have the following columns:


```
healthshed_id  year month day stat_1 stat_2 ... stat_n   
```


This means we should read in all of the exposures for a single timepoint at once. 
I think the smart thing to do is use a glob string to gather all of the pertinent files.
This will be the first function we export to the library:

In [0]:
#| echo: false
#| output: asis
show_doc(gather_exposure_geodataframes)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/publish.py#L28){target="_blank" style="float:right; font-size:smaller"}

### gather_exposure_geodataframes

>      gather_exposure_geodataframes (glob_string:str, polygon_id:str,
>                                     exposure:str)

*Read in a list of geo dataframes from the same time frame and merge them*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| glob_string | str | string for the path to search for the pertinent files |
| polygon_id | str | the string signifying the healthshed ID of the polygon |
| exposure | str | the exposure name |
| **Returns** | **list** |  |

In [ ]:
frames = here() / "data" / "testing" / "*madagascar*"

merged = gather_exposure_geodataframes(frames, "fs_uid", "2m_dewpoint_temperature")
merged[0].describe()

Processing files: 100%|##########| 3/3 [00:01<00:00,  1.82it/s]


stat,year,month,day,max,mean,min
count,254472.0,254472.000000,254472.000000,254472.000000,254472.000000,254472.000000
mean,2009.0,5.663043,15.836957,292.115845,290.383850,288.571930
std,0.0,3.701585,8.854244,3.794787,4.128042,4.721353
min,2009.0,1.000000,1.000000,277.250977,273.298462,268.284668
25%,2009.0,1.000000,8.000000,289.436615,287.414513,285.107178
50%,2009.0,6.000000,16.000000,292.382812,290.696609,288.960571
75%,2009.0,10.000000,23.250000,294.812210,293.349281,292.088867
max,2009.0,10.000000,31.000000,300.528076,299.109772,298.311462


This returns one file with all of the geometries and one file
with the statistics and exposures.

Now, with this, we can move on. The dataset was created in the UI and is available via search and test out how to upload it:

In [ ]:
resp = search_api.search("ERA5", data_type="dataset")

results = resp.json()['data']['items']

result = [x for x in results if "ERA5" in x['name']][0]
era5_pid = result['global_id']
result

{'name': 'ERA5 Exposure Aggregations for MDG nd NPL',
 'type': 'dataset',
 'url': 'https://doi.org/10.7910/DVN/MZO8HA',
 'global_id': 'doi:10.7910/DVN/MZO8HA',
 'description': 'This dataset contains daily aggregations of environmental exposures collected from the ERA5 Climate Data Store aggregated to geographic polygons in Madagascar and Nepal as part of the Climate-Smart Public Health project of the Golden Lab group at Harvard T.H. Chan School of Public Health. Exposures include 2-metre temperature, dewpoint temperature, total precipitation, and soil moisture. Aggregations include basic statistics per day within geospatial "healthsheds". The healthsheds\' geographies are provided directly by the respective governments.',
 'publisher': 'NSAPH',
 'citationHtml': 'Tapera, Tinashe, 2025, "ERA5 Exposure Aggregations for MDG nd NPL", <a href="https://doi.org/10.7910/DVN/MZO8HA" target="_blank">https://doi.org/10.7910/DVN/MZO8HA</a>, Harvard Dataverse, DRAFT VERSION',
 'identifier_of_dataver

We'll upload directly from file. In the case of ERA5 vs. LEGO, we
store the file on disk as LEGO hierarchy, but to upload it to dataverse
using a flat filename (since creating subdatasets to represent directories might be 
a bit of a hassle)

In [ ]:
# assuming the file has a path on disk like:
f_out = "environmental/exposures_era5/healthshed_daily/dewpoint_2024.parquet"
os.makedirs(here() / "data" / "testing" / os.path.dirname(f_out), exist_ok=True)
aggregations, geo = merged
aggregations.to_parquet(here() / "data" / "testing" / f_out, index=False)

datafile = Datafile()
datafile.set({
    # the id of the era5 dataset 
    "pid": era5_pid,
    # the path to the file on disk goes here
    "filename": str(here() / "data" / "testing" / f_out),
    # use the "label" to name the file
    "label": f_out.replace("/", "-")
})

In [ ]:
#| eval: false
resp = api.upload_datafile(era5_pid, str(here() / "data" / "testing" / f_out), datafile.json())

Pretty simple!

Now, we just need a main function to upload this data. The final upload is one file per
exposure per year, so these should be the variables we gather data for.

We should get some functionality to gather the groups of these files automatically, based on
the hydra config:

In [ ]:
target_dir = here() / "data" / "intermediate"

try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

cfg.development_mode = False
#cfg.query['year'] = 2017
#cfg.query['month'] = 11
#cfg.query['geography'] = "nepal"

In [0]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L321){target="_blank" style="float:right; font-size:smaller"}

### main

>      main (cfg:omegaconf.dictconfig.DictConfig)